# Классификатор релевантности патентов поверх  bge-m3

`query` и `title + abstract` → эмбеддинги из `patent-bi-encoder` (bge-m3) → признаки `[|u−v|, u⊙v, cos(u,v)]` → MLP → `not_relevant / uncertain / relevant`.

Рантайм: Colab, GPU T4 (нужна только для кодирования). Обучение головы занимает минуты.

## 0. Установка и пути

In [1]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.19.0" pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE = Path("bge-m3")
DATA_PATH = Path("labeled_pairs.parquet")
MODEL_DIR = BASE
WORK_DIR  = BASE / "work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

for p in [DATA_PATH, MODEL_DIR / "model.safetensors", MODEL_DIR / "modules.json",
          MODEL_DIR / "1_Pooling", MODEL_DIR / "2_Normalize"]:
    assert p.exists(), f"Не найдено: {p}"
print("Пути ок")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Пути ок


## 1. Проверка конфигурации модели (префиксы и нормализация)

In [6]:
for name in ["modules.json", "config_sentence_transformers.json", "sentence_bert_config.json"]:
    p = MODEL_DIR / name
    print(f"==== {name} ====")
    print(p.read_text() if p.exists() else "(нет файла)")

==== modules.json ====
[
  {
    "idx": 0,
    "name": "0",
    "path": "",
    "type": "sentence_transformers.base.modules.transformer.Transformer"
  },
  {
    "idx": 1,
    "name": "1",
    "path": "1_Pooling",
    "type": "sentence_transformers.sentence_transformer.modules.pooling.Pooling"
  },
  {
    "idx": 2,
    "name": "2",
    "path": "2_Normalize",
    "type": "sentence_transformers.base.modules.normalize.Normalize"
  }
]
==== config_sentence_transformers.json ====
{
  "__version__": {
    "pytorch": "2.14.1",
    "sentence_transformers": "6.1.0",
    "transformers": "5.19.0"
  },
  "default_prompt_name": null,
  "model_type": "SentenceTransformer",
  "prompts": {
    "document": "",
    "query": ""
  },
  "similarity_fn_name": "cosine"
}
==== sentence_bert_config.json ====
{
    "transformer_task": "feature-extraction",
    "modality_config": {
        "text": {
            "method": "forward",
            "method_output_name": "last_hidden_state"
        }
    },
    "modu

## 2. Данные

- убираем строки с ошибкой предсказания (нет `predicted_label`);
- сплит `challenge` не используем;
- классы: `not_relevant=0`, `uncertain=1`, `relevant=2`.

In [4]:
import numpy as np, pandas as pd, torch, random

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABELS = {"not_relevant": 0, "uncertain": 1, "relevant": 2}
ID2LABEL = {v: k for k, v in LABELS.items()}

df = pd.read_parquet(DATA_PATH)
df = df[(df["prediction_status"] == "success") & df["predicted_label"].isin(LABELS)]
df = df[df["split"].isin(["train", "validation", "test"])].reset_index(drop=True)
df["y"] = df["predicted_label"].map(LABELS)

print(len(df), "строк")
print(pd.crosstab(df["split"], df["predicted_label"]))

22601 строк
predicted_label  not_relevant  relevant  uncertain
split                                             
test                     1804       155        517
train                   12530       995       3762
validation               2120       174        544


## 3. Эмбеддинги (один раз, кэш на Drive)

Кодируем **уникальные** запросы (390) и **уникальные** патенты (~16.7k), а не 23k пар.

Текст патента: `"{title}. {abstract}"`, без очистки. **Формат должен совпадать с продакшеном**: если там текст собирается иначе, поменяйте `patent_text` и удалите кэш `.npy`.
Модель уже содержит слой `Normalize`, поэтому дополнительно ничего не нормализуем.

In [6]:
from sentence_transformers import SentenceTransformer

def patent_text(title, abstract):
    return f"{title}. {abstract}"

queries = df.drop_duplicates("query_id")[["query_id", "query"]].reset_index(drop=True)
patents = df.drop_duplicates("publication_id")[["publication_id", "title", "abstract"]].reset_index(drop=True)
patents["text"] = [patent_text(t, a) for t, a in zip(patents["title"], patents["abstract"])]
print("уникальных запросов:", len(queries), "| патентов:", len(patents))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer(str(MODEL_DIR), device=device)
model.max_seq_length = 512      # патенты длиннее ~512 токенов: 0.3%
print(model)

def encode_cached(texts, path, batch_size=32):
    if path.exists():
        emb = np.load(path)
        assert len(emb) == len(texts), "Кэш не совпадает с данными, удалите файл"
        return emb
    emb = model.encode(list(texts), batch_size=batch_size, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=False)
    emb = emb.astype("float32")
    np.save(path, emb)
    return emb

Q = encode_cached(queries["query"], WORK_DIR / "emb_queries.npy")
P = encode_cached(patents["text"],  WORK_DIR / "emb_patents.npy")

print(Q.shape, P.shape)
print("нормы запросов:", np.linalg.norm(Q, axis=1)[:3], "| патентов:", np.linalg.norm(P, axis=1)[:3])

уникальных запросов: 384 | патентов: 16308
device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
(384, 1024) (16308, 1024)
нормы запросов: [1. 1. 1.] | патентов: [0.99999994 1.         0.99999994]


## 4. Тензоры по сплитам

In [7]:
q_map = {k: i for i, k in enumerate(queries["query_id"])}
p_map = {k: i for i, k in enumerate(patents["publication_id"])}
df["q_idx"] = df["query_id"].map(q_map)
df["p_idx"] = df["publication_id"].map(p_map)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Qt = torch.tensor(Q, device=dev)
Pt = torch.tensor(P, device=dev)

def make_split(name):
    d = df[df["split"] == name]
    return (torch.tensor(d["q_idx"].values, device=dev),
            torch.tensor(d["p_idx"].values, device=dev),
            torch.tensor(d["y"].values, device=dev))

splits = {s: make_split(s) for s in ["train", "validation", "test"]}
y_np = {s: splits[s][2].cpu().numpy() for s in splits}

## 5. Модель

`features = [|u−v|, u⊙v, cos(u,v)]` (2049) → `Linear(256)` → ReLU → Dropout → `Linear(3)`.

In [8]:
import torch.nn as nn, torch.nn.functional as F

class PairHead(nn.Module):
    def __init__(self, dim=1024, hidden=256, n_classes=3, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * dim + 1, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, u, v):
        cos = F.cosine_similarity(u, v, dim=-1).unsqueeze(-1)
        return self.net(torch.cat([(u - v).abs(), u * v, cos], dim=-1))   # логиты

@torch.no_grad()
def predict_proba(m, split, bs=2048):
    m.eval()
    qi, pi, _ = splits[split]
    out = []
    for i in range(0, len(qi), bs):
        out.append(F.softmax(m(Qt[qi[i:i+bs]], Pt[pi[i:i+bs]]), dim=-1))
    return torch.cat(out).cpu().numpy()

## 6. Метрики и бейзлайн

Главная метрика: **PR-AUC (average precision) по классу `relevant`** против остальных. Также ROC-AUC, precision/recall/F1/F2 по `relevant` и macro-F1.
Бейзлайн: логистическая регрессия только на косинусе. Если голова его не обгоняет, она не нужна.

In [9]:
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_recall_fscore_support, f1_score, classification_report,
                             confusion_matrix, fbeta_score)
from sklearn.linear_model import LogisticRegression

def decide(probs, thr):
    # relevant, если p(relevant) >= thr, иначе argmax из not_relevant / uncertain
    pred = np.where(probs[:, 0] >= probs[:, 1], 0, 1)
    pred[probs[:, 2] >= thr] = 2
    return pred

def report(name, probs, y, thr):
    yb = (y == 2).astype(int)
    pred = decide(probs, thr)
    p, r, f1, _ = precision_recall_fscore_support(yb, (pred == 2).astype(int), average="binary", zero_division=0)
    print(f"[{name}] thr={thr:.3f} | PR-AUC(rel)={average_precision_score(yb, probs[:,2]):.4f} "
          f"ROC-AUC(rel)={roc_auc_score(yb, probs[:,2]):.4f} | relevant: P={p:.3f} R={r:.3f} "
          f"F1={f1:.3f} F2={fbeta_score(yb, (pred==2).astype(int), beta=2):.3f} | macro-F1={f1_score(y, pred, average='macro'):.3f}")

def best_f2_threshold(probs, y):
    yb = (y == 2).astype(int)
    p, r, t = precision_recall_curve(yb, probs[:, 2])
    f2 = 5 * p[:-1] * r[:-1] / np.maximum(4 * p[:-1] + r[:-1], 1e-12)
    return float(t[np.argmax(f2)])

# --- бейзлайн: LogReg на косинусе (relevant vs остальное)
def cos_feature(split):
    qi, pi, _ = splits[split]
    return F.cosine_similarity(Qt[qi], Pt[pi], dim=-1).cpu().numpy().reshape(-1, 1)

lr = LogisticRegression(class_weight="balanced").fit(cos_feature("train"), (y_np["train"] == 2).astype(int))
for s in ["validation", "test"]:
    sc = lr.predict_proba(cos_feature(s))[:, 1]
    yb = (y_np[s] == 2).astype(int)
    print(f"[baseline cos | {s}] PR-AUC(rel)={average_precision_score(yb, sc):.4f} "
          f"ROC-AUC={roc_auc_score(yb, sc):.4f} | доля relevant={yb.mean():.3f}")

[baseline cos | validation] PR-AUC(rel)=0.4125 ROC-AUC=0.8539 | доля relevant=0.061
[baseline cos | test] PR-AUC(rel)=0.3031 ROC-AUC=0.7811 | доля relevant=0.063


## 7. Обучение

Веса классов в loss (обратно пропорциональны частоте), AdamW, ранняя остановка по PR-AUC `relevant` на validation.

In [10]:
import copy

cfg = dict(hidden=256, dropout=0.3, lr=1e-3, weight_decay=1e-2, batch_size=256, max_epochs=60, patience=8)

counts = np.bincount(y_np["train"], minlength=3)
class_w = torch.tensor(len(y_np["train"]) / (3 * counts), dtype=torch.float32, device=dev)
print("train counts:", counts, "| веса классов:", class_w.cpu().numpy().round(2))

head = PairHead(dim=Q.shape[1], hidden=cfg["hidden"], dropout=cfg["dropout"]).to(dev)
opt = torch.optim.AdamW(head.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
loss_fn = nn.CrossEntropyLoss(weight=class_w)

qi_tr, pi_tr, y_tr = splits["train"]
n = len(y_tr)
best_ap, best_state, bad = -1, None, 0

for epoch in range(1, cfg["max_epochs"] + 1):
    head.train()
    perm = torch.randperm(n, device=dev)
    total = 0.0
    for i in range(0, n, cfg["batch_size"]):
        idx = perm[i:i + cfg["batch_size"]]
        loss = loss_fn(head(Qt[qi_tr[idx]], Pt[pi_tr[idx]]), y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(idx)
    pv = predict_proba(head, "validation")
    ap = average_precision_score((y_np["validation"] == 2).astype(int), pv[:, 2])
    print(f"epoch {epoch:02d} | train loss {total/n:.4f} | val PR-AUC(rel) {ap:.4f}")
    if ap > best_ap:
        best_ap, best_state, bad = ap, copy.deepcopy(head.state_dict()), 0
    else:
        bad += 1
        if bad >= cfg["patience"]:
            print("ранняя остановка"); break

head.load_state_dict(best_state)
print("лучший val PR-AUC(rel):", round(best_ap, 4))

train counts: [12530  3762   995] | веса классов: [0.46 1.53 5.79]
epoch 01 | train loss 1.0706 | val PR-AUC(rel) 0.2664
epoch 02 | train loss 0.9573 | val PR-AUC(rel) 0.2983
epoch 03 | train loss 0.8600 | val PR-AUC(rel) 0.2995
epoch 04 | train loss 0.8085 | val PR-AUC(rel) 0.3210
epoch 05 | train loss 0.7629 | val PR-AUC(rel) 0.3177
epoch 06 | train loss 0.7352 | val PR-AUC(rel) 0.3056
epoch 07 | train loss 0.7135 | val PR-AUC(rel) 0.2827
epoch 08 | train loss 0.6885 | val PR-AUC(rel) 0.2951
epoch 09 | train loss 0.6735 | val PR-AUC(rel) 0.2940
epoch 10 | train loss 0.6636 | val PR-AUC(rel) 0.2727
epoch 11 | train loss 0.6475 | val PR-AUC(rel) 0.2729
epoch 12 | train loss 0.6337 | val PR-AUC(rel) 0.2628
ранняя остановка
лучший val PR-AUC(rel): 0.321


## 8. Порог для `relevant` и итоговые метрики

Порог подбирается на **validation** по максимуму F2 (recall весит вдвое больше precision, так как для вас важнее найти подходящие патенты). Затем применяется к test без подгонки.

In [11]:
probs = {s: predict_proba(head, s) for s in ["validation", "test"]}
THR = best_f2_threshold(probs["validation"], y_np["validation"])
print("выбранный порог:", round(THR, 4), "\n")

for s in ["validation", "test"]:
    report(f"{s} | порог ", probs[s], y_np[s], thr=THR)

pred_test = decide(probs["test"], THR)
print("\n=== TEST, итоговое решение ===")
print(classification_report(y_np["test"], pred_test, target_names=list(LABELS), digits=3, zero_division=0))
print("confusion matrix (строки = истина, столбцы = прогноз):")
print(pd.DataFrame(confusion_matrix(y_np["test"], pred_test), index=list(LABELS), columns=list(LABELS)))

выбранный порог: 0.2376 

[validation | порог ] thr=0.238 | PR-AUC(rel)=0.3210 ROC-AUC(rel)=0.8121 | relevant: P=0.165 R=0.713 F1=0.268 F2=0.428 | macro-F1=0.396
[test | порог ] thr=0.238 | PR-AUC(rel)=0.2959 ROC-AUC(rel)=0.8292 | relevant: P=0.163 R=0.716 F1=0.265 F2=0.426 | macro-F1=0.422

=== TEST, итоговое решение ===
              precision    recall  f1-score   support

not_relevant      0.874     0.795     0.833      1804
   uncertain      0.366     0.108     0.167       517
    relevant      0.163     0.716     0.265       155

    accuracy                          0.647      2476
   macro avg      0.468     0.540     0.422      2476
weighted avg      0.724     0.647     0.658      2476

confusion matrix (строки = истина, столбцы = прогноз):
              not_relevant  uncertain  relevant
not_relevant          1434         91       279
uncertain              168         56       293
relevant                38          6       111


Для сравнения: обычный argmax по трём классам без порога.

In [12]:
for s in ["validation", "test"]:
    pr = probs[s].argmax(1)
    yb, pb = (y_np[s] == 2), (pr == 2)
    p, r, f1, _ = precision_recall_fscore_support(yb, pb, average="binary", zero_division=0)
    print(f"[{s} | чистый argmax 3 класса] relevant: P={p:.3f} R={r:.3f} F1={f1:.3f} | macro-F1={f1_score(y_np[s], pr, average='macro'):.3f}")

[validation | чистый argmax 3 класса] relevant: P=0.323 R=0.293 F1=0.307 | macro-F1=0.468
[test | чистый argmax 3 класса] relevant: P=0.294 R=0.387 F1=0.334 | macro-F1=0.535


## 9. Сохранение головы

In [13]:
import json

torch.save({"state_dict": head.state_dict(),
            "arch": {"dim": int(Q.shape[1]), "hidden": cfg["hidden"], "dropout": cfg["dropout"], "n_classes": 3}},
           WORK_DIR / "head.pt")

meta = {
    "labels": ID2LABEL,
    "threshold_relevant": THR,
    "decision_rule": "relevant if p_relevant >= threshold else argmax(p_not_relevant, p_uncertain)",
    "features": "[|u-v|, u*v, cos(u,v)]",
    "input": "u: query embedding, v: patent embedding (bge-m3, normalized, 1024)",
    "patent_text_format": "{title}. {abstract}",
    "train_config": cfg,
    "val_pr_auc_relevant": best_ap,
}
(WORK_DIR / "head_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("Сохранено в", WORK_DIR)

Сохранено в /content/drive/MyDrive/bge-m3/work
